# 01: Monolithic Student: walk-pick-carry-drop BC distillation plus DAgger fine-tune

| | |
|---|---|
| **Project**    | Humanoid Loco-Manipulation with the Unitree G1 |
| **Developer**  | Narayane Ribeiro Medeiros |
| **Repository** | [https://github.com/narayanerm/unitree-g1-humanoid-monolithic-policy](https://github.com/narayanerm/unitree-g1-humanoid-monolithic-policy) |

Extends 02_walk_pick to the full chain walk-pick-carry-drop (phase ids 0 to 3). Dataset comes from the batched 4-phase FSM `src.fsm_wpcd_batched` which reuses the walk_pick fixes (per-env A*, env_spacing=20m, per-env replica tables via spec_fn, _snap_pick_commands_batched at fire) and adds carry + drop fires without teleport.

### Run-Mode Flags

`REUSE_DATASET` skips the FSM data-gen; `REUSE_BC` skips both the dataset and the BC training (implies `REUSE_DATASET=True`); DAgger always runs. `DEBUG_LIGHT` shrinks everything to a 2-3 min sanity pass.

In [ ]:
DEBUG_LIGHT     = False   # True: n=8 envs + 3 epochs (~3 min end-to-end sanity)
REUSE_DATASET   = False   # skip FSM data-gen if outputs/<DATASET_NAME> already exists
REUSE_BC        = False   # skip BC and reuse latest g1_monolithic checkpoint; implies REUSE_DATASET=True

NUM_ENVS        = 8 if DEBUG_LIGHT else 8192
BASE_SEED       = 100
SPAWN_R_MIN     = 1.5
SPAWN_R_MAX     = 5.0
MAX_STEPS       = 1600

if REUSE_BC and not REUSE_DATASET:
    print('[config] REUSE_BC=True implies REUSE_DATASET=True; promoting.')
    REUSE_DATASET = True

DATASET_NAME = 'walk_pick_carry_drop_batched.npz'

### Environment Setup

Enforces Python 3.10 and sets headless MuJoCo (`MUJOCO_GL=egl`) before any mjlab import so the notebook runs on a remote GPU without a display.

In [ ]:
import os, sys
assert sys.version_info[:2] == (3, 10), f'Python 3.10 required, got {sys.version_info[:2]}'
os.environ.setdefault('MUJOCO_GL', 'egl')
sys.path.insert(0, os.getcwd())
print(f'cwd={os.getcwd()}')

### Dataset Generation

Calls `src.fsm_wpcd_batched.build_dataset_wpcd` at `NUM_ENVS` parallel envs. The 4-phase FSM runs walk (A* then brake) to pick (lift box) to carry (approach table 2 along the t1-side face) to drop (place box on table 2). Writes `outputs/<DATASET_NAME>` with `unified_obs`, `action`, `phase` arrays.

In [ ]:
import subprocess
from pathlib import Path

DATASET_LOCAL = Path(f'outputs/{DATASET_NAME}').resolve()
REPO_ROOT     = Path(os.getcwd()).resolve()

if REUSE_DATASET and DATASET_LOCAL.exists():
    print(f'[data] reusing existing {DATASET_LOCAL}')
else:
    print(f'[data] running 4-phase batched FSM (num_envs={NUM_ENVS}, max_steps={MAX_STEPS})')
    cmd = [
        sys.executable, '-u', '-m', 'src.fsm_wpcd_batched',
        '--num-envs', str(NUM_ENVS),
        '--seed', str(BASE_SEED),
        '--max-steps', str(MAX_STEPS),
        '--r-min', str(SPAWN_R_MIN),
        '--r-max', str(SPAWN_R_MAX),
        '--out', str(DATASET_LOCAL),
    ]
    sub_env = {**os.environ, 'MUJOCO_GL': 'egl', 'PYTHONUNBUFFERED': '1'}
    r = subprocess.run(cmd, cwd=str(REPO_ROOT), env=sub_env)
    if r.returncode != 0:
        raise RuntimeError(f'FSM builder failed with exit {r.returncode}')
    assert DATASET_LOCAL.exists(), f'builder returned 0 but {DATASET_LOCAL} was not written'

import numpy as np
blob = np.load(str(DATASET_LOCAL))
obs, act, phase = blob['unified_obs'], blob['action'], blob['phase']
print(f'dataset: {obs.shape[0]} transitions, obs={obs.shape[1]}, act={act.shape[1]}')
for pid in np.unique(phase):
    name = ['walk', 'pick', 'carry', 'drop'][int(pid)]
    n_p = int((phase == pid).sum())
    print(f'  phase {int(pid)} ({name}): {n_p} transitions ({n_p / obs.shape[0] * 100:.1f}%)')

### Dataset Diagnostics

Breaks down the per-phase row counts and prints the imbalance; the fsm_wpcd_batched module also prints end-of-run stats (fires per phase, drops_done, falls).

In [ ]:
import numpy as np
counts = np.array([(phase == i).sum() for i in range(4)], dtype=np.int64)
total = counts.sum()
print(f'[diag] total transitions: {total}')
for pid, name in enumerate(['walk', 'pick', 'carry', 'drop']):
    pct = 100 * counts[pid] / max(1, total)
    print(f'  phase {pid} ({name:>5s}): {counts[pid]:>8d}  ({pct:.1f}%)')
if counts[0] > 0 and counts[3] > 0:
    print(f'[diag] walk / drop ratio = {counts[0] / counts[3]:.1f} : 1')

### BC Training

Calls `train_bc` from `src.bc_train`. `balance_phases=True` applies a WeightedRandomSampler (inverse-frequency by phase) so the student sees walk / pick / carry / drop in equal proportion despite the physical imbalance. `REUSE_BC=True` recovers the latest checkpoint from `outputs/logs/g1_monolithic/`.

In [ ]:
from src.bc_train import train_bc, BCConfig

cfg = BCConfig(
    epochs=3 if DEBUG_LIGHT else 1000,
    batch_size=256 if DEBUG_LIGHT else 8192,
    lr=6e-4 if not DEBUG_LIGHT else 3e-4,
    balance_phases=True,
    # Walk+carry get equal share, pick+drop get a small boost (transition-rich phases).
    phase_target_weights=(0.2, 0.3, 0.2, 0.3),
)

if REUSE_BC:
    from src.utils import find_latest_checkpoint
    ck = find_latest_checkpoint('outputs/logs', experiment_hint='g1_monolithic')
    if ck is None:
        print('[train] no previous run found; training from scratch')
        ck = train_bc(DATASET_LOCAL, cfg=cfg)
    else:
        print(f'[train] reusing {ck}')
else:
    ck = train_bc(DATASET_LOCAL, cfg=cfg)

print(f'[train] BC checkpoint: {ck}')


### Held-out MSE Sanity

Loads the BC checkpoint and reports overall + per-phase MSE on the first 8192 transitions. Carry MSE is expected to look similar to pick MSE because both share the "hold box high" regime.

In [ ]:
import torch
from src.student_policy import MonolithicPolicyStudent

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
student = MonolithicPolicyStudent.load(ck, device=device)
student.eval()

n_eval = min(8192, obs.shape[0])
with torch.no_grad():
    x = torch.from_numpy(obs[:n_eval]).float().to(device)
    y = torch.from_numpy(act[:n_eval]).float().to(device)
    y_hat = student(x)
    mse_overall = torch.nn.functional.mse_loss(y_hat, y).item()
print(f'held-out BC MSE on {n_eval} transitions: {mse_overall:.5f}')

p = phase[:n_eval]
for pid in np.unique(p):
    name = ['walk', 'pick', 'carry', 'drop'][int(pid)]
    mask = p == pid
    if mask.sum() == 0:
        continue
    with torch.no_grad():
        mse_p = torch.nn.functional.mse_loss(y_hat[mask], y[mask]).item()
    print(f'  {name}: {int(mask.sum())} transitions, MSE={mse_p:.5f}')

### DAgger Fine-tune

Each round: rollout the current student in the WPCD env, label every state with the FSM oracle (walk/pick/carry/drop experts dispatched by the phase gate), and refit the student by MSE on the BC seed plus the last `buffer_window` rounds of labels.


In [ ]:
from src.dagger_finetune import DAggerConfig, dagger_finetune
from src.phase_gates import GateConfig

dagger_cfg = DAggerConfig(
    num_envs=256 if DEBUG_LIGHT else 1024,
    # Overshoot both rounds and epochs/round; no early stop. Grid fits 30 rounds within the B300 budget.
    rounds=2 if DEBUG_LIGHT else 25,
    episode_max_steps=200 if DEBUG_LIGHT else 1600,
    epochs_per_round=10 if DEBUG_LIGHT else 100,
    buffer_window=1 if DEBUG_LIGHT else 5,
    early_stop_patience=0,
    # Phase resets per round (sum <= 1.0; remainder is fresh walk spawn).
    pick_reset_fraction=0.05 if DEBUG_LIGHT else 0.10,
    carry_reset_fraction=0.10 if DEBUG_LIGHT else 0.30,
    drop_reset_fraction=0.05 if DEBUG_LIGHT else 0.10,
    # Match BC: boost pick+drop in the refit sampler.
    phase_target_weights=(0.2, 0.3, 0.2, 0.3),
    # Subsample BC seed in DEBUG_LIGHT so it fits a 4 GB VRAM GPU alongside the env.
    bc_seed_max_transitions=50_000 if DEBUG_LIGHT else 0,
    gate=GateConfig(
        spawn_r_min=SPAWN_R_MIN,
        spawn_r_max=SPAWN_R_MAX,
        seed=BASE_SEED,
    ),
    device=device,
)

dagger_ck = dagger_finetune(bc_ckpt=ck, bc_dataset=DATASET_LOCAL, cfg=dagger_cfg)
print(f'[dagger] fine-tuned checkpoint: {dagger_ck}')


### TensorBoard

BC curves live under `outputs/logs/g1_monolithic/`; DAgger curves under `outputs/logs/g1_monolithic_dagger/`. BC includes `Perf/eta_s`; DAgger logs per-round phase-fire counts under `Rollout/*` plus per-epoch `DAgger/train_loss`.


In [ ]:
# Launch TensorBoard from a terminal:
#   tensorboard --logdir_spec bc:outputs/logs/g1_monolithic,dagger:outputs/logs/g1_monolithic_dagger
